In [ ]:
#!nvidia-smi

In [ ]:
!pip install langchain
!pip install langchain_community
!pip install langchain_text_splitter
!pip install langchain_chroma
!pip install langchain_groq
!pip install langsmith
!pip install unstructured
!pip install unstructured-inference
!pip install "unstructured[pdf]"
!pip install sentence-transformers
!pip install pypdf
!pip install python-dotenv

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import UnstructuredFileIOLoader

In [ ]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Set environment variables directly
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

print("LangSmith tracing enabled successfully")

In [ ]:
import requests
url = "https://dspmuranchi.ac.in/pdf/Blog/python%20Built-in%20Functions.pdf"
response = requests.get(url)
with open("python_build_in_functions.pdf", "wb") as f:
  f.write(response.content)

In [ ]:
loader = UnstructuredFileIOLoader("python_built_in_functions.pdf")

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=400)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Load PDF
loader = PyPDFLoader("python_build_in_functions.pdf")
documents = loader.load()

# Add source metadata
for doc in documents:
    doc.metadata["source"] = "python_build_in_functions.pdf"

# Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

texts = text_splitter.split_documents(documents)

print("Pages:", len(documents))
print("Chunks:", len(texts))

In [ ]:
type(texts)

In [ ]:
texts[3]

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

In [ ]:
persist_directory = 'vector_db'

In [ ]:
from langchain_community.vectorstores import Chroma

vectordb = Chroma.from_documents(documents=texts, embedding=embeddings, persist_directory=persist_directory)
retriever = vectordb.as_retriever()

In [ ]:
llm = ChatGroq(model = 'llama-3.3-70b-versatile', temperature=7)

In [ ]:
vectordb = Chroma.from_documents(documents=texts, embedding=embeddings, persist_directory=persist_directory)

In [ ]:
llm = ChatGroq(model = 'llama-3.3-70b-versatile', temperature=2)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

In [ ]:
query = 'what are the function from the pdf'
response = qa_chain.invoke({'query': query})

In [ ]:
print(response['result'])

In [ ]:
query = 'what are the function from the pdf'
response = qa_chain.invoke({'query': query})
print(response['result'])
#print('*' * 30)
print('Source Document:', response['source_documents'][0].metadata['source'])

In [ ]:
query = 'what are the lambda function & explain filter map from the pdf'
response = qa_chain.invoke({'query': query})
print(response['result'])

In [32]:
import gradio as gr

# Chat Function
def chatbot_response(question):
    try:
        response = qa_chain.invoke({'query': question})
        answer = response["result"]
        sources = []
        for doc in response["source_documents"]:
            if "source" in doc.metadata:
                sources.append(doc.metadata["source"])
        source_text = "\n".join(list(set(sources)))
        final_response = f"""
{answer}

Source:
{source_text}
"""
        return final_response
    except Exception as e:
        return f"Error: {str(e)}"

# Gradio UI
demo = gr.Interface(
    fn=chatbot_response,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask questions from your pdf..."
    ),
    outputs="text",
    title="RAG PDF Chatbot",
    description="LLMOps project using LangChain + Groq + ChromaDB + Gradio",
    theme="soft"
)

demo.launch(debug=True)

Keyboard interruption in main thread... closing server.
